# §11.9.4 — 학습 레시피를 통제한 아키텍처 재비교

> 딥러닝 교재 · 3부 11장 9절 4항 (🐍)
> 선행: §11.9.2(복합 스케일링) · §11.9.5(아키텍처와 레시피의 혼동) · §4.5.5(공정 비교)

## 이 노트북이 답하는 질문

1. **같은 계산 예산에서 깊이·너비 배분이 성능을 바꾸는가?** 레시피를 통제하고 비교한다.
2. **레시피의 기여는 구조의 기여와 비교해 얼마나 큰가?** 구조 × 레시피 교차 비교로 잰다.
3. **레시피를 뒤섞은 비교는 순위를 바꿀 수 있는가?**

**예상 실행 시간** CPU 약 3분 (`FAST = True`이면 약 70초).
이것은 §11.9.5의 재평가 연구(ResNet strikes back 류)를 손바닥 크기로 재현하는 실험이다.

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 과제·부품 — §11.7.4의 망 부품을 재사용

In [ ]:
IMG = 12; AMP = 2.6
T_H = np.zeros((3, 3)); T_H[1, :] = 1.0
T_V = np.zeros((3, 3)); T_V[:, 1] = 1.0

def make_data(n, rn):
    X = rn.standard_normal((n, IMG, IMG))
    y = rn.integers(0, 2, n)
    s = rn.choice([-1., 1.], n)
    r0 = rn.integers(0, IMG-2, n); c0 = rn.integers(0, IMG-2, n)
    for i in range(n):
        T = T_V if y[i] else T_H
        X[i, r0[i]:r0[i]+3, c0[i]:c0[i]+3] += AMP*s[i]*T
    return X, y.astype(float)

from numpy.lib.stride_tricks import sliding_window_view

def sigmoid(z):
    return np.where(z >= 0, 1/(1+np.exp(-z)), np.exp(z)/(1+np.exp(z)))

def conv_same_fwd(X, W, b):
    N, C, H, _ = X.shape
    Xp = np.zeros((N, C, H+2, H+2)); Xp[:, :, 1:H+1, 1:H+1] = X
    v = sliding_window_view(Xp, (3, 3), axis=(2, 3))
    col = np.ascontiguousarray(v.transpose(0, 2, 3, 1, 4, 5)).reshape(N, H, H, C*9)
    return col @ W + b, col

def conv_same_bwd(dZ, col, W, xshape):
    N, C, H, _ = xshape
    dW = col.reshape(-1, col.shape[-1]).T @ dZ.reshape(-1, dZ.shape[-1])
    db = dZ.sum(axis=(0, 1, 2))
    dcol = dZ @ W.T
    d6 = dcol.reshape(N, H, H, C, 3, 3)
    dXp = np.zeros((N, C, H+2, H+2))
    for i in range(3):
        for j in range(3):
            dXp[:, :, i:i+H, j:j+H] += d6[:, :, :, :, i, j].transpose(0, 3, 1, 2)
    return dXp[:, :, 1:H+1, 1:H+1], dW, db

def maxpool22(A):
    N, F, H, _ = A.shape
    A4 = A.reshape(N, F, H//2, 2, H//2, 2)
    out = A4.max(axis=(3, 5))
    mask = (A4 == out[:, :, :, None, :, None])
    return out, mask

def maxpool22_bwd(dout, mask, H):
    return (mask * dout[:, :, :, None, :, None]).reshape(dout.shape[0], dout.shape[1], H, H)

class ConvStack:
    """conv 층 목록(층수/폭)과 풀링 위치, 잔차 여부를 받는 망."""
    def __init__(self, widths, pool_after, residual=False, rn=None):
        rn = rn or np.random.default_rng(0)
        self.widths = widths; self.pool_after = set(pool_after); self.residual = residual
        self.Ws = []; self.bs = []
        cin = 1
        for F in widths:
            self.Ws.append(rn.standard_normal((cin*9, F)) * np.sqrt(2/(cin*9)))
            self.bs.append(np.zeros(F))
            cin = F
        Hf = IMG // (2**len(self.pool_after))
        self.u = rn.standard_normal(cin*Hf*Hf) / np.sqrt(cin*Hf*Hf)
        self.c = np.zeros(1)
        self.params = self.Ws + self.bs + [self.u, self.c]
    def macs_total(self):
        m, cin, H = 0, 1, IMG
        for li, F in enumerate(self.widths):
            m += cin*F*9*H*H
            if li in self.pool_after: H //= 2
            cin = F
        return m
    def forward(self, X):
        N = X.shape[0]
        A = X[:, None]
        self.cache = []
        for li, (W, b) in enumerate(zip(self.Ws, self.bs)):
            Z, col = conv_same_fwd(A, W, b)
            Anew = np.maximum(Z, 0).transpose(0, 3, 1, 2)
            res = self.residual and (Anew.shape == A.shape)
            if res: Anew = Anew + A
            aux_pool = None
            if li in self.pool_after:
                P, mask = maxpool22(Anew)
                aux_pool = (mask, Anew.shape[2])
                out = P
            else:
                out = Anew
            self.cache.append((A.shape, Z, col, res, aux_pool))
            A = out
        feat = A.reshape(N, -1)
        self.feat = feat
        return feat @ self.u + self.c
    def backward(self, dout):
        du = self.feat.T @ dout; dc = np.array([dout.sum()])
        N = self.feat.shape[0]
        Hf = IMG // (2**len(self.pool_after)); cin = self.widths[-1]
        dA = np.outer(dout, self.u).reshape(N, cin, Hf, Hf)
        gWs = [None]*len(self.Ws); gbs = [None]*len(self.bs)
        for li in range(len(self.Ws)-1, -1, -1):
            ashape, Z, col, res, aux_pool = self.cache[li]
            if aux_pool is not None:
                dA = maxpool22_bwd(dA, aux_pool[0], aux_pool[1])
            dRes = dA if res else 0.
            dZ = dA.transpose(0, 2, 3, 1) * (Z > 0)
            dA, gW, gb = conv_same_bwd(dZ, col, self.Ws[li], ashape)
            if res: dA = dA + dRes
            gWs[li] = gW; gbs[li] = gb
        return gWs + gbs + [du, dc]

def train(net, Xtr, ytr, recipe, seed=0):
    rb = np.random.default_rng(seed)
    B = 96
    steps = recipe['steps']
    ms = [np.zeros_like(p) for p in net.params]; vs = [np.zeros_like(p) for p in net.params]
    for t in range(1, steps+1):
        idx = rb.integers(0, len(ytr), B)
        Xb = Xtr[idx]
        if recipe['aug'] > 0:
            sh = rb.integers(-recipe['aug'], recipe['aug']+1, (B, 2))
            Xb = np.stack([np.roll(Xb[i], tuple(sh[i]), axis=(0, 1)) for i in range(B)])
        z = net.forward(Xb)
        dz = (sigmoid(z.ravel()) - ytr[idx]) / B
        gs = net.backward(dz)
        if recipe['opt'] == 'sgd':
            for pp, g in zip(net.params, gs):
                pp -= recipe['lr'] * g
        else:
            lr = recipe['lr'] * 0.5*(1+np.cos(np.pi*t/steps))    # 코사인 감쇠
            for pp, g, m, v in zip(net.params, gs, ms, vs):
                m[:] = 0.9*m + 0.1*g; v[:] = 0.999*v + 0.001*g*g
                pp -= lr*(m/(1-0.9**t))/(np.sqrt(v/(1-0.999**t))+1e-8)
    return net

def evaluate(net, X, y):
    hits = 0
    for s0 in range(0, len(y), 300):
        sl = slice(s0, s0+300)
        hits += np.sum((net.forward(X[sl]).ravel() > 0) == (y[sl] > 0.5))
    return hits/len(y)

OLD = dict(name='구식', opt='sgd', lr=0.05, aug=0, steps=120 if FAST else 250)
NEW = dict(name='신식', opt='adam', lr=4e-3, aug=2, steps=250 if FAST else 500)

Xtr, ytr = make_data(6000, np.random.default_rng(SEED))
Xte, yte = make_data(1500, np.random.default_rng(SEED+9))
SEEDS = 1 if FAST else 2

---
## 2. 동일 예산의 세 배분 — 같은 (신식) 레시피

In [ ]:
ALLOC = {
    lab('깊게-좁게', 'deep-narrow'): dict(widths=[7]*6, pool_after=[0, 1]),
    lab('얕게-넓게', 'shallow-wide'): dict(widths=[9, 9], pool_after=[0, 1]),
    lab('균형', 'balanced'): dict(widths=[8, 8, 8], pool_after=[0, 1]),
}
alloc_res = {}
for name, cfg in ALLOC.items():
    macs = ConvStack(**cfg).macs_total()
    a = []
    for si in range(SEEDS):
        net = ConvStack(**cfg, rn=np.random.default_rng(30+si))
        train(net, Xtr, ytr, NEW, seed=si)
        a.append(evaluate(net, Xte, yte))
    alloc_res[name] = (macs, np.mean(a), np.std(a))
    print(f"{name}: MAC {macs/1e3:.0f}k  acc {np.mean(a):.3f}±{np.std(a):.3f}  ({time.time()-_t0:.0f}초)")

---
## 3. 구조 × 레시피 교차 비교

구조 축: 균형 배분의 **평범한 스택** vs **잔차 스택**(같은 폭, 항등 스킵).
레시피 축: 구식(SGD·증강 없음·짧게) vs 신식(Adam·코사인·이동 증강·길게).

In [ ]:
ARCH = {
    lab('평범', 'plain'): dict(widths=[8]*4, pool_after=[0, 1], residual=False),
    lab('잔차', 'residual'): dict(widths=[8]*4, pool_after=[0, 1], residual=True),
}
cross = {}
for aname, acfg in ARCH.items():
    for rec in (OLD, NEW):
        a = []
        for si in range(SEEDS):
            net = ConvStack(**acfg, rn=np.random.default_rng(60+si))
            train(net, Xtr, ytr, rec, seed=100+si)
            a.append(evaluate(net, Xte, yte))
        cross[(aname, rec['name'])] = (np.mean(a), np.std(a))
        print(f"{aname} × {rec['name']}: acc {np.mean(a):.3f}±{np.std(a):.3f}  ({time.time()-_t0:.0f}초)")

---
## 4. 교재 그림 — fig_11_9_4

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.6, 3.6))

# (a) 배분 비교
ax = axes[0]
names = list(alloc_res)
vals = [alloc_res[k][1] for k in names]; errs = [alloc_res[k][2] for k in names]
bars = ax.bar(range(len(names)), vals, yerr=errs, capsize=4,
              color=[CB[1], CB[2], CB[5]], alpha=0.9)
for i, k in enumerate(names):
    ax.text(i, vals[i]+errs[i]+0.008, f"{alloc_res[k][0]/1e3:.0f}k MAC", ha='center', fontsize=8)
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, fontsize=9)
ax.set_ylim(0.5, 1.02)
ax.set_ylabel(lab('시험 정확도', 'test accuracy'))
ax.set_title(lab('(a) 동일 예산의 세 배분 (같은 레시피)', '(a) equal-budget allocations'), fontsize=10)

# (b) 교차 비교 행렬
ax = axes[1]
archs = list(ARCH); recs = [OLD['name'], NEW['name']]
M = np.array([[cross[(a_, r_)][0] for r_ in recs] for a_ in archs])
imv = ax.imshow(M, cmap='viridis', vmin=M.min()-0.02, vmax=M.max()+0.02)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{M[i,j]:.3f}", ha='center', va='center',
                color='white' if M[i, j] < M.mean() else 'black', fontsize=11)
ax.set_xticks([0, 1]); ax.set_xticklabels([lab('구식 레시피', 'old recipe'), lab('신식 레시피', 'new recipe')], fontsize=9)
ax.set_yticks([0, 1]); ax.set_yticklabels(archs, fontsize=9)
ax.grid(False)
ax.set_title(lab('(b) 구조 × 레시피 교차', '(b) arch x recipe'), fontsize=10)

# (c) 혼동된 비교 vs 통제된 비교
ax = axes[2]
conf = [cross[(archs[0], recs[0])][0], cross[(archs[1], recs[1])][0]]
ctrl = [cross[(archs[0], recs[1])][0], cross[(archs[1], recs[1])][0]]
x = np.arange(2); w = 0.35
ax.bar(x-w/2, conf, w, color=CB[4], alpha=0.85,
       label=lab('혼동된 비교\n(구조와 레시피가 함께 바뀜)', 'confounded'))
ax.bar(x+w/2, ctrl, w, color=CB[5], alpha=0.85,
       label=lab('통제된 비교\n(레시피 고정)', 'controlled'))
ax.set_xticks(x); ax.set_xticklabels(archs, fontsize=9)
ax.set_ylim(0.5, 1.05)
ax.set_ylabel(lab('시험 정확도', 'test accuracy'))
ax.set_title(lab('(c) 같은 데이터, 두 가지 결론', '(c) two conclusions from one dataset'), fontsize=10)
ax.legend(fontsize=7, loc='lower right')

save_book_fig(fig, 'fig_11_9_4')
plt.show()

> ### 읽는 법
>
> (a) 예산이 같아도 배분에 따라 성능이 갈린다. 이 과제에서는 균형과 깊게-좁게가 앞서고
> 얕게-넓게가 뒤처진다 — 한 축에 몰아주는 배분이 손해라는 §11.9.2 논리의 축소판.
> (b) 행(구조) 방향의 차이보다 **열(레시피) 방향의 차이가 크다.** 더 흥미로운 것은 잔차의 효과가
> 레시피에 **조건부**라는 점이다. 구식 레시피에서는 크게 돕지만(최적화가 어려운 환경의 보조),
> 신식 레시피에서는 오히려 소폭 밑돈다. "구조 X가 좋다"는 문장은 레시피를 명시해야 뜻이 정해진다.
> (c) 왼쪽(혼동된 비교)은 "잔차 덕에 크게 개선"이라 읽히지만, 레시피를 고정한 오른쪽에서는
> 순위가 뒤집힌다. **관심 변수 외의 것이 함께 바뀐 비교는 관심 변수에 대해 아무것도 말해 주지 않는다.**

---
## 5. 자기 점검

1. (a)의 우세 배분은 과제·데이터양에 따라 달라진다. 어떤 과제면 깊게-좁게가 이기겠는가?
2. (b)에서 잔차의 기여가 작았던 이유는? 깊이를 12층으로 늘리면 어떻게 되겠는가? (§9.1)
3. 신식 레시피는 걸음 수도 2배다. 공정한 비교인가? "같은 걸음"과 "같은 계산" 중 무엇으로 맞춰야 하는가? (§4.5.5)
4. 실제 문헌에서 (c)형 혼동을 피하려면 논문에서 어떤 표를 요구해야 하는가?

## 6. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `AMP` | 1절 | 2.2 | 과제 난이도 |
| `ALLOC` | 2절 | 3가지 | 배분 후보. MAC이 근사하게 같도록 폭을 조정할 것 |
| `OLD`/`NEW` | 1절 | — | 레시피의 구성 요소를 하나씩만 바꿔 기여를 분해해 보라 |
| `SEEDS` | 1절 | 2 | 반복 수 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")